# Notebook 07 — Análisis de Errores

## Objetivo

Análisis manual de errores del mejor modelo tradicional (Exp. 1) y, si está disponible, DistilBERT (Exp. 3) sobre el subconjunto político. Taxonomía fija según ADR Experimento 5.

- **FP (Falso Positivo):** noticia real clasificada como fake (`label=0`, `pred=1`)
- **FN (Falso Negativo):** noticia fake clasificada como real (`label=1`, `pred=0`)

> El notebook exporta plantillas CSV en blanco; las categorías y comentarios se completan **a mano** antes de graficar.

In [1]:
import warnings

warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlp.error_analysis import (
    ERROR_CATEGORIES,
    annotations_complete,
    build_error_frame,
    category_distribution,
    confidence_from_proba,
    load_annotations,
    model_confidence,
    warn_if_small_sample,
    write_blank_annotation_template,
)
from nlp.paths import *
from nlp.plotting import save_figure, setup_style

setup_style()

BASELINE_ANNOTATIONS = RESULTS_ERROR / "error_annotations_baseline.csv"
TRANSFORMER_ANNOTATIONS = RESULTS_ERROR / "error_annotations_transformer.csv"
DISTILBERT_DIR = RESULTS_MODELS / "best_distilbert"


## 1. Cargar modelo y generar predicciones

In [2]:
from nlp.modeling import get_text_col

pol_test = pd.read_csv(DATA_PROCESSED / "politics_test.csv")
pipe = joblib.load(RESULTS_MODELS / "best_baseline_politics.joblib")

config_path = RESULTS_MODELS / "best_baseline_politics_config.json"
if config_path.exists():
    best_cfg = pd.read_json(config_path, typ="series")
    TEXT_COL = get_text_col(best_cfg["text_field"], best_cfg["stopwords"])
else:
    TEXT_COL = "clean_full_text_without_stopwords"

X_test = pol_test[TEXT_COL].fillna("")
y_true = pol_test["label"]
y_pred = pipe.predict(X_test)
scores = model_confidence(pipe, X_test)

baseline_errors = build_error_frame(pol_test, y_true, y_pred, scores)
n_fp = int((baseline_errors["error_type"] == "FP").sum())
n_fn = int((baseline_errors["error_type"] == "FN").sum())
print(f"FP={n_fp}, FN={n_fn}")
warn_if_small_sample(len(baseline_errors))
baseline_errors[["title", "error_type", "confidence"]].head()


FP=6, FN=11
Total errores en test: 17
ADVERTENCIA: el ADR sugiere ≥30 errores, pero el modelo solo comete 17 en test. Se analizan todos los casos disponibles; esto limita la generalización cualitativa.


,title,error_type,confidence
1752,‘BRIEFCASES FULL OF MONEY’…Uranium One Underco...,FN,4.475142
308,Why John McCain Will Never Vote to Repeal and ...,FN,2.392720
2494,Democratic U.S. senator seeks audit of EPA chi...,FP,1.345398
1764,FBI UNDERCOVER Informant On Hillary’s 2010 Sal...,FN,0.799081
2420,WATCH: “IT’S THE MOST WONDERFUL TIME OF THE YE...,FN,0.688171


## 2. Plantilla para anotación manual

Se exportan **todos** los errores del baseline en test. Completar a mano en `results/error_analysis/error_annotations_baseline.csv`:

1. Leer `title` + `text_fragment` de cada fila.
2. Asignar `category` con **una** de las categorías del ADR:
   - `lenguaje_neutral_en_fake`, `titulo_ambiguo`, `ironia_sarcasmo`, `informacion_parcialmente_verdadera`, `sesgo_fuente`, `tema_politico_cargado`, `otro`
3. Escribir una observación breve en `comment`.

Si el archivo ya existe, **no se sobrescribe** (protege anotaciones previas).

In [3]:
write_blank_annotation_template(baseline_errors, BASELINE_ANNOTATIONS)

print("Categorías válidas (ADR):")
for cat in ERROR_CATEGORIES:
    print(f"  - {cat}")


Plantilla creada: /Users/lucasrouco/workspace/ITBA/NLP/nlp/results/error_analysis/error_annotations_baseline.csv (17 filas)
Categorías válidas (ADR):
  - lenguaje_neutral_en_fake
  - titulo_ambiguo
  - ironia_sarcasmo
  - informacion_parcialmente_verdadera
  - sesgo_fuente
  - tema_politico_cargado
  - otro


In [5]:
if BASELINE_ANNOTATIONS.exists():
    pd.read_csv(BASELINE_ANNOTATIONS).head(10)
else:
    print("Completar el CSV y volver a ejecutar las secciones 3+.")


## 3. Distribución de categorías de error

In [6]:
if not BASELINE_ANNOTATIONS.exists():
    print("Sin plantilla baseline. Ejecutar sección 2 primero.")
elif not annotations_complete(load_annotations(BASELINE_ANNOTATIONS)):
    print(
        "Anotaciones incompletas. Completar 'category' en "
        f"{BASELINE_ANNOTATIONS} antes de graficar."
    )
else:
    baseline_ann = load_annotations(BASELINE_ANNOTATIONS)
    cat_dist = category_distribution(baseline_ann)
    cat_dist.to_csv(
        RESULTS_ERROR / "error_category_distribution_baseline.csv",
        index=False,
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=cat_dist, x="count", y="category", ax=ax, color="#3498db")
    ax.set_title("Distribución de categorías de error — baseline (test)")
    ax.set_xlabel("Cantidad")
    save_figure(fig, RESULTS_FIGURES / "07_error_category_distribution_baseline.png")
    plt.show()
    cat_dist


## 4. Comparación con transformer (condicional)

Si `results/models/best_distilbert` existe, se generan predicciones y una plantilla de anotación para sus errores. También se compara F2 agregado cuando hay `transformer_results.csv`.

In [ ]:
transformer_results_path = RESULTS_METRICS / "transformer_results.csv"
best_tr = None

if transformer_results_path.exists():
    tr_res = pd.read_csv(transformer_results_path)
    best_tr = tr_res.sort_values("f2_fake", ascending=False).iloc[0]
    print("Mejor transformer (test):")
    print(best_tr[["model", "f2_fake", "accuracy", "recall_fake"]].to_string())

    baseline_df = pd.read_csv(RESULTS_METRICS / "baseline_results.csv")
    if "split" in baseline_df.columns:
        baseline_df = baseline_df[baseline_df["split"] == "test"]
    politics_baseline = baseline_df[baseline_df["dataset_scope"] == "politics"]
    baseline_best = politics_baseline.sort_values("f2_fake", ascending=False).iloc[0]

    compare_df = pd.DataFrame(
        [
            {"model_type": "baseline", "f2_fake": baseline_best["f2_fake"]},
            {"model_type": "transformer", "f2_fake": best_tr["f2_fake"]},
        ]
    )
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.barplot(data=compare_df, x="model_type", y="f2_fake", ax=ax)
    ax.set_title("F2 fake: baseline vs transformer (test)")
    save_figure(fig, RESULTS_FIGURES / "07_error_analysis_baseline_vs_transformer.png")
    plt.show()
else:
    print("transformer_results.csv no disponible; ejecutar notebook 05.")

if not DISTILBERT_DIR.exists():
    print("DistilBERT no entrenado; correr notebook 05 para errores del transformer.")
else:
    import torch
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
    )

    from nlp.embeddings import embedding_text_series
    from nlp.features import load_source_normalization_decision
    from nlp.io import load_splits
    from nlp.transformers_data import NewsDataset, prepare_transformer_texts

    EMBEDDING_COLS = ["label", "title", "text", "clean_full_text_without_stopwords"]
    _, _, pol_test_tr = load_splits("politics", columns=EMBEDDING_COLS)

    source_decision = load_source_normalization_decision(SOURCE_ABLATION_DECISION)
    normalize_source = source_decision["use_source_normalization"]

    te_texts = prepare_transformer_texts(
        embedding_text_series(
            pol_test_tr,
            "clean_full_text_without_stopwords",
            normalize_source=normalize_source,
        )
    )

    tokenizer = AutoTokenizer.from_pretrained(str(DISTILBERT_DIR))
    model = AutoModelForSequenceClassification.from_pretrained(str(DISTILBERT_DIR))
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    max_length = 256
    if best_tr is not None and pd.notna(best_tr.get("max_length")):
        max_length = int(best_tr["max_length"])

    test_ds = NewsDataset(te_texts, pol_test_tr["label"], tokenizer, max_length)
    pred = Trainer(model=model, data_collator=data_collator).predict(test_ds)
    y_tr_pred = np.argmax(pred.predictions, axis=-1)
    y_proba_fake = torch.softmax(torch.tensor(pred.predictions), dim=-1)[:, 1].numpy()
    tr_scores = confidence_from_proba(y_tr_pred, y_proba_fake)

    transformer_errors = build_error_frame(
        pol_test_tr,
        pol_test_tr["label"],
        y_tr_pred,
        tr_scores,
    )
    n_fp_tr = int((transformer_errors["error_type"] == "FP").sum())
    n_fn_tr = int((transformer_errors["error_type"] == "FN").sum())
    print(f"Errores transformer: total={len(transformer_errors)} (FP={n_fp_tr}, FN={n_fn_tr})")
    warn_if_small_sample(len(transformer_errors))

    write_blank_annotation_template(transformer_errors, TRANSFORMER_ANNOTATIONS)

    baseline_ready = (
        BASELINE_ANNOTATIONS.exists()
        and annotations_complete(load_annotations(BASELINE_ANNOTATIONS))
    )
    transformer_ready = (
        TRANSFORMER_ANNOTATIONS.exists()
        and annotations_complete(load_annotations(TRANSFORMER_ANNOTATIONS))
    )

    if baseline_ready and transformer_ready:
        baseline_dist = category_distribution(load_annotations(BASELINE_ANNOTATIONS))
        transformer_dist = category_distribution(
            load_annotations(TRANSFORMER_ANNOTATIONS)
        )
        baseline_dist["model_type"] = "baseline"
        transformer_dist["model_type"] = "transformer"
        compare_dist = pd.concat([baseline_dist, transformer_dist], ignore_index=True)
        compare_dist.to_csv(
            RESULTS_ERROR / "error_category_distribution_comparison.csv",
            index=False,
        )

        fig, ax = plt.subplots(figsize=(10, 5))
        sns.barplot(
            data=compare_dist,
            x="count",
            y="category",
            hue="model_type",
            ax=ax,
        )
        ax.set_title("Categorías de error: baseline vs transformer")
        ax.set_xlabel("Cantidad")
        save_figure(fig, RESULTS_FIGURES / "07_error_category_comparison.png")
        plt.show()
    else:
        print(
            "Comparación cualitativa pendiente: completar anotaciones en "
            f"{BASELINE_ANNOTATIONS.name} y {TRANSFORMER_ANNOTATIONS.name}."
        )


Mejor transformer (test):
model          distilbert-base-uncased
f2_fake                       0.997337
accuracy                      0.998892
recall_fake                   0.996674


## Conclusiones

- La taxonomía del ADR guía la lectura cualitativa; completar `error_annotations_baseline.csv` antes de interpretar patrones.
- Si hay **menos de 30 errores** en test (alto desempeño del modelo), el análisis es ilustrativo pero no exhaustivo.
- El análisis manual **no es 100% reproducible**; se mitiga con taxonomía fija y CSV versionable.
- Los FN de alta confianza (fake con estilo periodístico) son los más preocupantes para un despliegue de apoyo a verificación humana.
- Tras entrenar DistilBERT (notebook 05), repetir la anotación en `error_annotations_transformer.csv` y comparar distribuciones.

## 5. Consolidación de resultados

In [7]:

from nlp.metrics import consolidate_results

all_results = consolidate_results(
    baseline_path=RESULTS_METRICS / 'baseline_results.csv',
    output_path=RESULTS_METRICS / 'all_model_results.csv',
)
print(f'Resultados consolidados: {len(all_results)} filas')
all_results.head(10)


Resultados consolidados: 3 filas


,accuracy,precision_fake,recall_fake,f1_fake,f2_fake,roc_auc,model,vectorizer,text_field,stopwords,ngram_range,max_features,best_param,dataset_scope,split,source_condition,use_source_normalization
0,0.996760,0.980854,0.991071,0.985936,0.989011,0.999748,linear_svm,bow,full,with_stopwords,"(1, 2)",30000.0,1.0,full_dataset,test,NaN,NaN
1,0.993722,0.993311,0.987805,0.990550,0.988901,0.997920,linear_svm,bow,body,without_stopwords,"(1, 2)",10000.0,10.0,politics,test,NaN,NaN
2,0.950148,0.956005,0.891353,0.922547,0.903574,0.980430,linguistic_features_lr,NaN,title_text,NaN,NaN,NaN,10.0,politics,test,NaN,False
